# Fetch Company Numbers from Companies House API

This notebook reads businesses from `business_input.csv` and fetches their company number and company ID using the Companies House API.

In [19]:
# Import Required Libraries
import pandas as pd
import requests
import os

In [20]:
# Load Business Input CSV
input_path = "business_input.csv"
df = pd.read_csv(input_path)
print(f"Loaded {len(df)} businesses from {input_path}")
df.head()

Loaded 39 businesses from business_input.csv


,Company Name
0,Aston Vaughan
1,Knight & Knoxley
2,David & Co
3,Dean & Co
4,Weatherill Property Group


In [21]:
# Configure Companies House API
# Get API key from environment variable or set directly
# Sign up for free at: https://developer.companieshouse.gov.uk/developer/applications

api_key = os.environ.get("COMPANIES_HOUSE_API_KEY", "")
if not api_key:
    api_key = input("Enter your Companies House API key: ").strip()

if not api_key:
    raise ValueError("API key is required. Get one from https://developer.companieshouse.gov.uk/")

# Set up authentication
auth = requests.auth.HTTPBasicAuth(api_key, "")
base_url = "https://api.companieshouse.gov.uk"

headers = {
    "Accept": "application/json"
}

print(f"API configured successfully for {base_url}")

API configured successfully for https://api.companieshouse.gov.uk


In [22]:
# Fetch Company Details
from urllib.parse import quote_plus

# Real Estate SIC codes for filtering
# 68310 - Real estate agencies
# 68320 - Management of real estate on a fee or contract basis
# 68100 - Buying and selling of own real estate
# 68201 - Renting and operating of own or leased real estate
# 68209 - Other letting and operating of own or leased real estate
REAL_ESTATE_SIC_CODES = ["68310", "68320", "68100", "68201", "68209"]

def get_company_profile(company_number, auth, base_url, headers):
    """Fetch full company profile from Companies House API"""
    profile_url = f"{base_url}/company/{company_number}"
    try:
        response = requests.get(profile_url, auth=auth, headers=headers, timeout=30)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        print(f"    Error fetching profile: {e}")
        return None

def search_company(company_name, auth, base_url, headers, require_real_estate=True):
    """Search for a company by name using Companies House API
    Args:
        company_name: Name of the company to search
        auth: Authentication credentials
        base_url: API base URL
        headers: Request headers
        require_real_estate: If True, only return results with real estate SIC codes
    """
    search_url = f"{base_url}/search/companies?q={quote_plus(company_name)}"
    
    try:
        response = requests.get(search_url, auth=auth, headers=headers, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            items = data.get("items", [])
            
            if not items:
                return None
            
            # If real estate filtering is required, fetch full profile for each match
            if require_real_estate:
                for result in items:
                    company_number = result.get("company_number", "")
                    if not company_number:
                        continue
                    
                    # Fetch full company profile to get SIC codes
                    profile = get_company_profile(company_number, auth, base_url, headers)
                    if profile:
                        sic_codes = profile.get("sic_codes", [])
                        # Check if any SIC code starts with real estate codes
                        if any(sic.startswith(tuple(REAL_ESTATE_SIC_CODES)) for sic in sic_codes):
                            return {
                                "company_number": company_number,
                                "company_id": company_number,
                                "company_name": profile.get("company_name", ""),
                                "company_status": profile.get("company_status", ""),
                                "company_type": profile.get("company_type", ""),
                                "sic_codes": sic_codes,
                                "address": profile.get("registered_office_address", {}),
                                "incorporation_date": profile.get("incorporation_date", "")
                            }
                # No real estate match found
                return None
            else:
                # Return first match without filtering
                first_result = items[0]
                return {
                    "company_number": first_result.get("company_number", ""),
                    "company_id": first_result.get("company_number", ""),
                    "company_name": first_result.get("title", ""),
                    "company_status": first_result.get("company_status", ""),
                    "company_type": first_result.get("company_type", ""),
                    "sic_codes": first_result.get("sic_codes", []),
                    "address": first_result.get("address", {}),
                    "match_score": first_result.get("matches", {}).get("score", 0)
                }
        elif response.status_code == 401:
            print("Authentication error. Check your API key.")
            return None
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None
    except Exception as e:
        print(f"Exception: {e}")
        return None

# Process each business
results = []

for idx, row in df.iterrows():
    company_name = row.get("Company Name", "")
    print(f"[{idx + 1}/{len(df)}] Searching: {company_name}")
    
    result = search_company(company_name, auth, base_url, headers, require_real_estate=True)
    
    if result:
        results.append({
            "Company Name": company_name,
            "Company Number": result["company_number"],
            "Company ID": result["company_id"],
            "Company Status": result["company_status"],
            "Company Type": result["company_type"],
            "SIC Codes": ", ".join(result.get("sic_codes", [])),
            "Address": result["address"]
        })
        print(f"    Found: {result['company_number']} (SIC: {result.get('sic_codes', [])})")
    else:
        results.append({
            "Company Name": company_name,
            "Company Number": "",
            "Company ID": "",
            "Company Status": "",
            "Company Type": "",
            "SIC Codes": "",
            "Address": {}
        })
        print(f"    No real estate company found")

print(f"\nProcessed {len(results)} businesses")

[1/39] Searching: Aston Vaughan
    Found: 15510783 (SIC: ['68209', '68310', '68320', '96090'])
[2/39] Searching: Knight & Knoxley
    Found: 09678002 (SIC: ['68310'])
[3/39] Searching: David & Co
    Found: 05202234 (SIC: ['68310', '68320'])
[4/39] Searching: Dean & Co
    Found: 14795756 (SIC: ['68209'])
[5/39] Searching: Weatherill Property Group
    Found: 11397540 (SIC: ['68310'])
[6/39] Searching: Pearson Keehan
    Found: 13189050 (SIC: ['68310'])
[7/39] Searching: Hamlyn Smith
    Found: 13163237 (SIC: ['68310'])
[8/39] Searching: Smith & Co Sussex
    Found: 16264838 (SIC: ['68310'])
[9/39] Searching: Nicholas James Property
    Found: 14770162 (SIC: ['68100'])
[10/39] Searching: John Hoole
    No real estate company found
[11/39] Searching: Phillips & Still
    Found: 08757928 (SIC: ['68310'])
[12/39] Searching: Justin Lloyd
    No real estate company found
[13/39] Searching: Spencer & Leigh
    Found: 04794913 (SIC: ['68310'])
[14/39] Searching: Mishon Mackay
    Found: 0644

In [23]:
# Save Results to CSV
results_df = pd.DataFrame(results)

# Format address for CSV
def format_address(addr):
    if not addr:
        return ""
    parts = [addr.get("address_line_1", ""), addr.get("address_line_2", ""), 
             addr.get("locality", ""), addr.get("postal_code", "")]
    return ", ".join([p for p in parts if p])

# Create output with required columns
output_df = pd.DataFrame({
    "Company Name": results_df["Company Name"],
    "Company House Number": results_df["Company Number"],
    "Company House URL": results_df["Company Number"].apply(
        lambda x: f"https://find-and-update.company-information.service.gov.uk/company/{x}" if x else ""
    )
})

output_path = "company_numbers_output.csv"
output_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"\nSummary:")
print(f"  Total businesses: {len(output_df)}")
print(f"  Found (Real Estate): {len(output_df[output_df['Company House Number'] != ''])}")
print(f"  Not found: {len(output_df[output_df['Company House Number'] == ''])}")
output_df

Results saved to company_numbers_output.csv

Summary:
  Total businesses: 39
  Found (Real Estate): 37
  Not found: 2


,Company Name,Company House Number,Company House URL
0,Aston Vaughan,15510783,https://find-and-update.company-information.se...
1,Knight & Knoxley,09678002,https://find-and-update.company-information.se...
2,David & Co,05202234,https://find-and-update.company-information.se...
3,Dean & Co,14795756,https://find-and-update.company-information.se...
4,Weatherill Property Group,11397540,https://find-and-update.company-information.se...
5,Pearson Keehan,13189050,https://find-and-update.company-information.se...
6,Hamlyn Smith,13163237,https://find-and-update.company-information.se...
7,Smith & Co Sussex,16264838,https://find-and-update.company-information.se...
8,Nicholas James Property,14770162,https://find-and-update.company-information.se...
9,John Hoole,,


In [24]:
output_df.to_clipboard()